# Exercise 4 · Branch, pull request, merge

You will make a change on a branch, merge it into `main` with a pull request on GitHub, and pull the result.

**Goal:** Commit on a branch, push it, open a pull request into your own `main`, merge it on GitHub, and pull `main`.

**Why it matters:** Coding agents work on branches and open pull requests. You need to know what a branch and a merge are to follow them.

**Before you start:**
- You did exercises 1 and 3 today: you have your own copy on GitHub, and `git push` works in your terminal.
- Step 2 clones your copy into this exercise's `practice/starter` with one command: `git clone <your link> starter`.
- A branch is a separate line of commits. `main` is your copy's main branch.

**What the check measures:** your local `main` holds a pull request's merge commit with your branch's commit, and matches GitHub's `main`.

**Done when:** the last cell prints `Done:`.

## The situation

You want to add a notes file to your copy. You make it on a branch, then merge it the way a team does: with a pull request. The note is made up.

## Step 1 · Write your link

Run the cell with your link from exercise 1. It makes `practice/` and prints the clone command for Step 2. It does not go online.

Expect a line that starts with `git clone https://github.com/`.

In [ ]:
# Step 1: paste the link you handed in for exercise 1 between the quotes, then run this cell
MY_REPO = ""

import re
from pathlib import Path

PRACTICE = Path("practice")
PRACTICE.mkdir(exist_ok=True)
found = re.search(r"github\.com/([\w.-]+)/([\w.-]+)", MY_REPO)
if (PRACTICE / "starter").exists():
    print("Kept practice/starter: it is already there. Go to Step 3.")
elif not found:
    print("MY_REPO is empty or not a GitHub link. Paste your link from exercise 1, then run this cell again.")
else:
    print("In Step 2, clone your copy with this line:")
    print(f"git clone https://github.com/{found[1]}/{found[2].removesuffix('.git')}.git starter")

## Step 2 · Clone your copy

Choose **Terminal > New Terminal**. Type each line, then press Enter. For the third line, paste the line that Step 1 printed:

```sh
source .venv/bin/activate
cd week-02/day-02/exercise-04-branch-pull-request-merge/practice
git clone https://github.com/<your-account>/<name>.git starter
cd starter
git log --oneline
```

Expect `Add a line to README` at the top: your commit from exercise 3.

## Step 3 · Commit on a new branch

```sh
git switch -c add-notes
echo "# Notes for challenge 1" > NOTES.md
git add NOTES.md
git commit -m "Add NOTES.md"
```

Expect `Switched to a new branch 'add-notes'`, then a line with `[add-notes` and `Add NOTES.md`.

## Step 4 · Push the branch

```sh
git push -u origin add-notes
```

`origin` is the name Git gives the repository you cloned from: your copy on GitHub. `-u` links your branch to the GitHub branch, so a plain `git pull` knows where to read.

Expect a line that ends with `add-notes -> add-notes`.

### Learn: a pull request asks to merge one branch into another

A pull request shows a branch's commits before they join `main`. With **Merge pull request**, "all commits from the feature branch are added to the base branch in a merge commit" ([Pull request merges](https://docs.github.com/en/pull-requests/reference/pull-request-merges)). The base branch is the one that receives them.

## Step 5 · Open a pull request and merge it

1. Open your copy on github.com. In the yellow banner, click **Compare & pull request**. If there is no banner, click the **Pull requests** tab. Then click **New pull request**, and choose `add-notes` to compare with `main`.
2. Check the two branch menus at the top: **base** shows `main`, and **compare** shows `add-notes`. Click **Create pull request**.
3. At the bottom, click **Merge pull request**. If you do not see it, open the merge dropdown and select **Create a merge commit**. Then click **Confirm merge**.

Expect the pull request to show that it is merged.

<details><summary>Show the answer: which merge option?</summary>

**Create a merge commit**. Why: it keeps your branch's commit and adds a merge commit. By default its message starts `Merge pull request #` ([Configuring commit merging](https://docs.github.com/en/repositories/configuring-branches-and-merges-in-your-repository/configuring-pull-request-merges/configuring-commit-merging-for-pull-requests)). Squash and rebase make no merge commit.

</details>

## Step 6 · Pull the merged main

```sh
git switch main
git pull
git log --oneline -3
```

Expect, newest first: `Merge pull request #1 from <your-account>/add-notes`, then `Add NOTES.md`. Your number can differ.

## Start over (only if you need it)

In the terminal, run `cd ..` to go to `practice/`. Then run `mv starter starter-old` to rename the clone. Then do Step 2 again. In Step 3 and Step 4, use a new branch name.

## Check your work

Run the cell. Expect three `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do. The last check asks GitHub, so it needs the internet.

In [ ]:
# Check your work: find the pull request's merge commit on main, then compare main with GitHub (a public read)
import os
import re
import subprocess
from pathlib import Path

CLONE = Path("practice") / "starter"
NO_PROMPT = dict(os.environ, GIT_TERMINAL_PROMPT="0")


def git(*args) -> str:
    """Return git's output in practice/starter, or "" if the command fails."""
    try:
        done = subprocess.run(["git", "-C", str(CLONE), *args], capture_output=True, text=True,
                              timeout=60, env=NO_PROMPT)
    except (OSError, subprocess.TimeoutExpired):
        return ""
    return done.stdout.strip() if done.returncode == 0 else ""


def brings_in_a_commit(line: str) -> bool:
    """True when this merge commit's second parent adds at least one commit to the first."""
    parents = line.split("\t")[1].split()
    return len(parents) >= 2 and git("rev-list", "--count", f"{parents[0]}..{parents[1]}") not in ("", "0")


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    if not (CLONE / ".git").exists():
        return "practice/starter is missing. Clone your copy into it (Step 2)."
    owner = re.search(r"github\.com[/:]([\w.-]+)/", git("config", "--get", "remote.origin.url"))
    if not owner or owner[1].lower() == "gitrolltraining":
        return "practice/starter is not a clone of your copy on github.com. Clone your own link (Start over, Step 2)."
    print(f"ok: practice/starter is a clone of your copy ({owner[1]})")
    merges = git("log", "refs/heads/main", "--merges", "--format=%H%x09%P%x09%s").splitlines()
    merged = [line for line in merges
              if line.split("\t")[2].startswith("Merge pull request #") and brings_in_a_commit(line)]
    if not merged:
        return ("your local main has no merge commit from a pull request. Merge it on GitHub (Step 5), "
                "then run git switch main and git pull (Step 6).")
    print(f"ok: main holds {merged[0].split(chr(9))[2]!r}")
    remote = git("ls-remote", "origin", "refs/heads/main").split()
    if not remote:
        return "Git could not read your copy on GitHub. Check your internet connection, then run this cell again."
    if remote[0] != git("rev-parse", "refs/heads/main"):
        return "your local main and GitHub's main differ. Run git switch main and git pull (Step 6)."
    print("ok: your main and GitHub's main are the same")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Do Step 2, then run this cell."
print(f"Not yet: {problem}" if problem else
      "Done: your branch reached main through a pull request, and your clone has the merged main.")

## What you learned

- A branch keeps work apart from `main` until you merge it.
- A pull request shows the branch's commits. Merging it adds a merge commit to `main` on GitHub. `git pull` brings that commit to your computer.

**Reflect:** An agent opens a pull request for you. What do you read before you click **Merge pull request**?